## Feature Engineering & Business Metrics
### E-Commerce 360 Analytics

---

### Purpose of this Notebook
In this notebook, we transform the cleaned base tables into **business-ready analytical features** based on our findings from EDA (`03_eda.ipynb`).

### Key Features Created
1. **Delivery SLA & Logistics:** `delivery_delay_days`, `is_late`, and `sla_status` (On-Time vs Late Delivery).
2. **English Product Categories:** Translated Portuguese categories to clean English title case (e.g. `bed_bath_table` $\rightarrow$ `Bed Bath Table`).
3. **Brazilian State Full Names & Macro-Regions:** Mapped 2-letter state codes (`SP`, `RJ`, `MG`, etc.) to full names and the 5 official Brazilian regions (`Southeast`, `South`, `Northeast`, `Central-West`, `North`).
4. **Order Basket Metrics:** `order_value`, `order_freight`, `order_total`, and `order_item_count`.
5. **Customer RFM Segmentation:** Recency (days), Frequency (orders), Monetary (spend), and Customer Segments (`Champions`, `Loyal`, `At Risk`, `Hibernating`, etc.).
6. **Seller & Product Aggregations:** Total GMV, units sold, and average order value per vendor/product.

---

In [1]:
# Step 1: Import Libraries and Load Base Tables
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_path = Path("../data/processed")

order_dates = [
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]

customers = pd.read_csv(data_path / "customers.csv")
orders = pd.read_csv(data_path / "orders.csv", parse_dates=order_dates)
order_items = pd.read_csv(data_path / "order_items.csv")
order_payments = pd.read_csv(data_path / "order_payments.csv")
order_reviews = pd.read_csv(data_path / "order_reviews.csv")
products = pd.read_csv(data_path / "products.csv")
sellers = pd.read_csv(data_path / "sellers.csv")
category_translation = pd.read_csv(data_path / "product_category_name_translation.csv")

print("Base tables loaded successfully.")

Base tables loaded successfully.


## 1. Order Date and Time Features
Extract year, month, quarter, and day of week to support trend analysis in SQL and Power BI.

In [2]:
# 1. Order Date / Time Extraction
orders["order_year"] = orders["order_purchase_timestamp"].dt.year
orders["order_month"] = orders["order_purchase_timestamp"].dt.month
orders["order_year_month"] = orders["order_purchase_timestamp"].dt.strftime("%Y-%m")
orders["order_quarter"] = orders["order_purchase_timestamp"].dt.quarter
orders["order_day_of_week"] = orders["order_purchase_timestamp"].dt.day_name()
orders["order_hour"] = orders["order_purchase_timestamp"].dt.hour
orders["is_weekend"] = orders["order_purchase_timestamp"].dt.dayofweek.isin([5, 6]).astype(int)

print("Order date features created successfully.")
orders[["order_purchase_timestamp", "order_year", "order_month", "order_day_of_week"]].head()

Order date features created successfully.


,order_purchase_timestamp,order_year,order_month,order_day_of_week
0,2017-10-02 10:56:33,2017,10,Monday
1,2018-07-24 20:41:37,2018,7,Tuesday
2,2018-08-08 08:38:49,2018,8,Wednesday
3,2017-11-18 19:28:06,2017,11,Saturday
4,2018-02-13 21:18:39,2018,2,Tuesday


## 2. Delivery Turnaround & Logistics SLA Features
Calculate actual delivery duration, estimated delivery duration, and flag orders that breached promised delivery deadlines.

In [3]:
# 2. Delivery Turnaround & SLA Status
orders["actual_delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["estimated_delivery_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.days
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

# Binary flag (1 = Late, 0 = On-Time)
orders["is_late"] = np.where(orders["delivery_delay_days"] > 0, 1, 0)
orders["sla_status"] = np.where(orders["is_late"] == 1, "Late Delivery", "On-Time")

print("Delivery SLA features created successfully.")
orders[["actual_delivery_days", "delivery_delay_days", "is_late", "sla_status"]].describe()

Delivery SLA features created successfully.


,actual_delivery_days,delivery_delay_days,is_late
count,96476.000000,96476.000000,99441.000000
mean,12.094086,-11.876881,0.065717
std,9.551746,10.183854,0.247789
min,0.000000,-147.000000,0.000000
25%,6.000000,-17.000000,0.000000
50%,10.000000,-12.000000,0.000000
75%,15.000000,-7.000000,0.000000
max,209.000000,188.000000,1.000000


## 3. Product Categories in English & Product Performance Metrics
Translate Portuguese category names to readable English title case, and calculate total sales and units sold per product.

In [4]:
# 3.1 Product Category English Translation
products = products.merge(category_translation, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna("Other")

# Format category in Title Case with spaces (e.g. Bed Bath Table)
products["category_english"] = products["product_category_name_english"].str.replace("_", " ").str.title()

# 3.2 Product Sales Aggregates from order_items
prod_agg = order_items.groupby("product_id").agg(
    product_revenue=("price", "sum"),
    product_units_sold=("order_item_id", "count"),
    product_order_count=("order_id", "nunique")
).reset_index()

products = products.merge(prod_agg, on="product_id", how="left")
products[["product_revenue", "product_units_sold", "product_order_count"]] = products[["product_revenue", "product_units_sold", "product_order_count"]].fillna(0)

print("Products enriched with English categories and sales metrics.")
products[["product_id", "category_english", "product_revenue", "product_units_sold"]].head()

Products enriched with English categories and sales metrics.


,product_id,category_english,product_revenue,product_units_sold
0,1e9e8ef04dbcff4541ed26657ea517e5,Perfumery,10.91,1
1,3aa071139cb16b67ca9e5dea641aaa2f,Art,248.00,1
2,96bd76ec8810374ed1b65e291975717f,Sports Leisure,79.80,1
3,cef67bcfe19066a932b7673e239eb23d,Baby,112.30,1
4,9dc1a7de274444849c219cff195d0b71,Housewares,37.90,1


## 4. Brazilian State Full Names & Macro-Regions in English
Map 2-letter state abbreviations (`SP`, `RJ`, `MG`, etc.) to full state names and the 5 official Brazilian Geographic Macro-Regions for both customers and sellers.

In [5]:
# 4.1 State Full Names & Macro-Region Mappings
state_full_names = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapa", "AM": "Amazonas", "BA": "Bahia",
    "CE": "Ceara", "DF": "Distrito Federal", "ES": "Espirito Santo", "GO": "Goias",
    "MA": "Maranhao", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul", "MG": "Minas Gerais",
    "PA": "Para", "PB": "Paraiba", "PR": "Parana", "PE": "Pernambuco", "PI": "Piaui",
    "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul",
    "RO": "Rondonia", "RR": "Roraima", "SC": "Santa Catarina", "SP": "Sao Paulo",
    "SE": "Sergipe", "TO": "Tocantins"
}

state_regions_english = {
    "SP": "Southeast", "RJ": "Southeast", "MG": "Southeast", "ES": "Southeast",
    "PR": "South", "RS": "South", "SC": "South",
    "BA": "Northeast", "CE": "Northeast", "PE": "Northeast", "MA": "Northeast",
    "PB": "Northeast", "RN": "Northeast", "AL": "Northeast", "PI": "Northeast", "SE": "Northeast",
    "DF": "Central-West", "GO": "Central-West", "MT": "Central-West", "MS": "Central-West",
    "AM": "North", "PA": "North", "RO": "North", "TO": "North", "AC": "North", "AP": "North", "RR": "North"
}

# Apply to Customers
customers["customer_state_name"] = customers["customer_state"].map(state_full_names)
customers["customer_region"] = customers["customer_state"].map(state_regions_english)

# Apply to Sellers
sellers["seller_state_name"] = sellers["seller_state"].map(state_full_names)
sellers["seller_region"] = sellers["seller_state"].map(state_regions_english)

# 4.2 Seller Sales Aggregates
seller_agg = order_items.groupby("seller_id").agg(
    seller_revenue=("price", "sum"),
    seller_units_sold=("order_item_id", "count"),
    seller_order_count=("order_id", "nunique")
).reset_index()

seller_agg["seller_avg_order_value"] = (seller_agg["seller_revenue"] / seller_agg["seller_order_count"]).round(2)
sellers = sellers.merge(seller_agg, on="seller_id", how="left").fillna(0)

print("Customers and Sellers enriched with English region and state names.")
customers[["customer_city", "customer_state", "customer_state_name", "customer_region"]].head()

Customers and Sellers enriched with English region and state names.


,customer_city,customer_state,customer_state_name,customer_region
0,franca,SP,Sao Paulo,Southeast
1,sao bernardo do campo,SP,Sao Paulo,Southeast
2,sao paulo,SP,Sao Paulo,Southeast
3,mogi das cruzes,SP,Sao Paulo,Southeast
4,campinas,SP,Sao Paulo,Southeast


## 5. Order Value and Basket Features
Calculate the total merchandise spend, shipping freight, basket item count, and payment method for each order.

In [6]:
# 5.1 Order Basket Aggregates from order_items
order_basket = order_items.groupby("order_id").agg(
    order_value=("price", "sum"),
    order_freight=("freight_value", "sum"),
    order_item_count=("order_item_id", "count")
).reset_index()

order_basket["order_total"] = order_basket["order_value"] + order_basket["order_freight"]

# 5.2 Primary Payment Method and Installments from order_payments
order_pay = order_payments.groupby("order_id").agg(
    payment_type_primary=("payment_type", lambda x: x.mode()[0] if not x.empty else "unknown"),
    payment_installments_max=("payment_installments", "max"),
    payment_value_total=("payment_value", "sum")
).reset_index()

# Merge onto Orders
orders = orders.merge(order_basket, on="order_id", how="left")
orders = orders.merge(order_pay, on="order_id", how="left")

print("Orders enriched with basket values and payment features.")
orders[["order_id", "order_value", "order_freight", "order_total", "payment_type_primary"]].head()

Orders enriched with basket values and payment features.


,order_id,order_value,order_freight,order_total,payment_type_primary
0,e481f51cbdc54678b7cc49136f2d6af7,29.99,8.72,38.71,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,118.70,22.76,141.46,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,159.90,19.22,179.12,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,45.00,27.20,72.20,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,19.90,8.72,28.62,credit_card


## 6. Customer RFM Segmentation
Calculate Recency ($R$), Frequency ($F$), and Monetary ($M$) metrics for each unique customer (`customer_unique_id`) to identify customer loyalty tiers.

In [7]:
# 6.1 Join Delivered Orders with Customers
cust_orders = orders[orders["order_status"] == "delivered"].merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner")

# Snapshot date: 1 day after the latest purchase
snapshot_date = cust_orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

# 6.2 Calculate R, F, M per Unique Customer
rfm = cust_orders.groupby("customer_unique_id").agg(
    recency_days=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("order_total", "sum")
).reset_index()

# 6.3 RFM Scoring (1 to 5 Quantile Cuts)
rfm["r_score"] = pd.qcut(rfm["recency_days"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
# For frequency, since 97% have 1 order, we score directly: 1 order = 1, 2 orders = 3, 3+ orders = 5
rfm["f_score"] = np.where(rfm["frequency"] >= 3, 5, np.where(rfm["frequency"] == 2, 3, 1))
rfm["m_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Composite RFM Score
rfm["rfm_score"] = rfm["r_score"] * 100 + rfm["f_score"] * 10 + rfm["m_score"]

# 6.4 Assign Plain English Customer Segments
def map_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 3 and m >= 4:
        return "Champions"
    elif f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f == 1:
        return "Recent Buyers"
    elif r <= 2 and m >= 4:
        return "At Risk"
    elif r <= 2 and f == 1 and m <= 2:
        return "Hibernating"
    else:
        return "Promising"

rfm["customer_segment"] = rfm.apply(map_segment, axis=1)

# Merge RFM onto Customers table
customers = customers.merge(rfm[["customer_unique_id", "recency_days", "frequency", "monetary", "customer_segment"]], on="customer_unique_id", how="left")
customers["customer_segment"] = customers["customer_segment"].fillna("New / Inactive")

print("Customer RFM segments created successfully.")
customers["customer_segment"].value_counts()

Customer RFM segments created successfully.


customer_segment
Recent Buyers      36300
Promising          25344
Hibernating        15334
At Risk            13762
Loyal Customers     3772
New / Inactive      2758
Champions           2171
Name: count, dtype: int64

## 7. Save Feature-Engineered Datasets
Save the enriched tables into `data/processed/` ready for SQL ingestion and Power BI dashboards.

In [8]:
# 7. Save Enriched Datasets
orders.to_csv(data_path / "orders_feature_engineered.csv", index=False)
customers.to_csv(data_path / "customers_feature_engineered.csv", index=False)
products.to_csv(data_path / "products_feature_engineered.csv", index=False)
sellers.to_csv(data_path / "sellers_feature_engineered.csv", index=False)

print("All 4 feature-engineered datasets saved successfully to data/processed/")

# Verification
saved_files = [
    "orders_feature_engineered.csv",
    "customers_feature_engineered.csv",
    "products_feature_engineered.csv",
    "sellers_feature_engineered.csv"
]

for f in saved_files:
    file_path = data_path / f
    print(f"✓ {f}: Exists = {file_path.exists()} | Rows = {pd.read_csv(file_path).shape[0]:,}")

All 4 feature-engineered datasets saved successfully to data/processed/
✓ orders_feature_engineered.csv: Exists = True | Rows = 99,441
✓ customers_feature_engineered.csv: Exists = True | Rows = 99,441
✓ products_feature_engineered.csv: Exists = True | Rows = 32,951
✓ sellers_feature_engineered.csv: Exists = True | Rows = 3,095


---
## 8. Summary of Feature Engineering

| Table | New Features Added | Business Value |
|---|---|---|
| **Orders** | `delivery_delay_days`, `is_late`, `sla_status`, `order_total`, `payment_type_primary`, `order_year_month` | Powers Logistics SLA tracking and financial revenue metrics. |
| **Customers** | `customer_state_name`, `customer_region` (English), `recency_days`, `frequency`, `monetary`, `customer_segment` | Powers customer RFM loyalty targeting and regional geographic analysis. |
| **Products** | `category_english` (Title Case English), `product_revenue`, `product_units_sold` | Replaces Portuguese names with readable English categories across all reports. |
| **Sellers** | `seller_state_name`, `seller_region`, `seller_revenue`, `seller_avg_order_value` | Powers seller performance tiers and Pareto concentration analysis. |

**Next Step in Project:**  
The enriched datasets are now certified and ready to be loaded into the **3NF SQL Database (`data/ecommerce.db`)** and the **Power BI Dashboard**!